# 🎨 Notebook 1: Facebook (core social graph) — Class Design


## 🛠️ Setup

```bash
cd 07-object-oriented-design/facebook
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we are designing

The *core* of a social network: **users, friendships, posts, comments, reactions, and a news feed**.
We deliberately skip photos, groups, chat, ads — so the model stays small enough to reason about on a whiteboard.

### Domain model (ASCII)

```
    User ──friends── User          (symmetric: if A friends B, B friends A)
     │
     │ authors
     ▼
    Post ──◆── Comment
     │
     │ ◇────── Reaction (LIKE, LOVE, HAHA, WOW, SAD, ANGRY)
     ▼
   NewsFeed  (derived per-user from friends' posts)
```

### 5 design decisions we'll justify with code below

1. **Friendship is symmetric** — represented as a `set[User]` mirrored on both sides.
2. **Reactions are an enum**, not a subclass hierarchy — they differ only in value, not behavior.
3. **One reaction per user per post** — stored as `dict[user_id → ReactionType]` so re-reacting overwrites.
4. **News feed is derived**, not stored — `feed(user)` = posts by `user ∪ friends`, sorted by time.
5. **IDs are surrogate integers**, not object identity — so we can hash users in sets safely.


## 🚦 Bad → Best progression

This notebook walks through **four common mistakes** in modeling a social network, each with *runnable* code that
demonstrates why the naive choice breaks down. Notebook 2 then puts the winning choices together in one clean implementation.


### ❌ Bad #1 — Asymmetric friendship (one-sided `add`)

*Naive*: call `a.friends.append(b)` and forget that `b` also has to list `a`. Now `are_friends(a, b) != are_friends(b, a)`.
In real systems this shows up as "I accepted their request but they still can't see my posts."


In [ ]:
class BadUser:
    def __init__(self, name):
        self.name = name
        self.friends = []          # list, not symmetric

    def add_friend(self, other):
        self.friends.append(other) # only mutates SELF

a = BadUser('Alice'); b = BadUser('Bob')
a.add_friend(b)
print('A sees B as friend:', b in a.friends)   # True
print('B sees A as friend:', a in b.friends)   # False — bug!


### ✅ Fix — mirror the edge on the other side, and use a `set` (O(1) membership, no duplicates)


In [ ]:
class GoodUser:
    def __init__(self, name):
        self.name = name
        self.friends: set['GoodUser'] = set()

    def add_friend(self, other):
        if other is self:          # never friend yourself
            return
        self.friends.add(other)
        other.friends.add(self)    # symmetric — this is the whole point

a = GoodUser('Alice'); b = GoodUser('Bob')
a.add_friend(b)
a.add_friend(b)                    # idempotent because of set semantics
assert b in a.friends and a in b.friends
assert len(a.friends) == 1
print('symmetric and idempotent OK')


### ❌ Bad #2 — Reaction as a subclass hierarchy

*Naive*: create `class LikeReaction`, `class LoveReaction`, … — one subclass per emoji.
This is **overengineered**: they all carry the same data (who reacted, when), and differ only in *value*.
Adding a new reaction type now means adding a new class, writing `isinstance` checks, etc.


In [ ]:
class Reaction:            # abstract base
    def __init__(self, user): self.user = user
class LikeReaction(Reaction):  pass
class LoveReaction(Reaction):  pass
class HahaReaction(Reaction):  pass
# ... and on and on. To count likes you'd do isinstance checks. Yuck.
reactions = [LikeReaction('alice'), LoveReaction('bob'), LikeReaction('carol')]
like_count = sum(1 for r in reactions if isinstance(r, LikeReaction))
print('likes (isinstance way):', like_count)


### ✅ Fix — `ReactionType` as an `Enum`

All reaction types behave identically; only the *label* differs. An enum says exactly that.


In [ ]:
from enum import Enum
class ReactionType(Enum):
    LIKE  = '👍'
    LOVE  = '❤️'
    HAHA  = '😂'
    WOW   = '😮'
    SAD   = '😢'
    ANGRY = '😡'

reactions = [('alice', ReactionType.LIKE),
             ('bob',   ReactionType.LOVE),
             ('carol', ReactionType.LIKE)]
like_count = sum(1 for _, r in reactions if r is ReactionType.LIKE)
print('likes (enum way):', like_count)


### ❌ Bad #3 — Storing reactions as a `list` (allows duplicates per user)

*Naive*: every click appends to a list, so Alice liking twice counts as 2 likes.
Facebook's rule is **one reaction per user per post** (clicking LOVE *replaces* your LIKE).


In [ ]:
reactions_list = []
reactions_list.append(('alice', 'LIKE'))
reactions_list.append(('alice', 'LOVE'))   # user changed their mind
reactions_list.append(('alice', 'LOVE'))   # accidental double-click
print('list version — alice counted multiple times:', reactions_list)


### ✅ Fix — `dict[user_id → ReactionType]`

Assigning to the same key **replaces** the old reaction — exactly the behavior we want, in one line.


In [ ]:
reactions = {}              # user_id -> ReactionType
reactions[1] = 'LIKE'       # alice likes
reactions[1] = 'LOVE'       # alice changes to love (overwrites)
reactions[1] = 'LOVE'       # double click — still one entry
print('dict version — alice appears once:', reactions)


### ❌ Bad #4 — Precomputing and storing every user's feed on every post

*Naive*: keep a `feed: list[Post]` on every `User` and push into all friends' feeds on every post.
Fine when you have 10 users. For a celebrity with 10M followers, one post = 10M writes. This is the
classic **fanout-on-write vs fanout-on-read** tradeoff.

For an educational lab, the *best* default is **fanout-on-read**: compute the feed from the graph when asked.
Simple, always consistent, no duplicate storage. We note fanout-on-write as an optimization (see Notebook 2, Step 8).


In [ ]:
# Pseudocode of the two strategies (don't run — just read)
# fanout-on-WRITE:  post(user, p) -> for f in user.friends: f.feed.insert(0, p)    # O(#friends) writes per post
# fanout-on-READ :  feed(user)    -> [p for p in all_posts if p.author in {user}|user.friends]   # O(#posts) per view
print('Rule of thumb:')
print(' - Celebrity with millions of followers -> fanout-on-read for THEIR posts')
print(' - Normal user -> fanout-on-write is fine')
print(' - Real systems (Facebook, Twitter) use a HYBRID.')


## 📋 Final class table

| Class | Role | Key fields |
|---|---|---|
| `User` | person in the graph | `id`, `name`, `friends: set[User]`, `blocked: set[User]` |
| `Post` | something a user wrote | `author`, `content`, `ts`, `privacy`, `comments`, `reactions` |
| `Comment` | reply on a post | `author`, `text`, `ts` |
| `ReactionType` | enum of allowed reactions | `LIKE, LOVE, HAHA, WOW, SAD, ANGRY` |
| `Privacy` | enum of visibility levels | `PUBLIC, FRIENDS, ONLY_ME` |
| `NewsFeed` | derives each user's feed from the graph | `for_user(u)` |

➡️ Continue to **Notebook 2** for the runnable implementation with privacy, blocking, mutual friends, and a fanout-on-write bonus.
